In [10]:
import psycopg2
import pandas as pd

In [11]:
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [12]:
cur.execute('SELECT * FROM "30pid_superfamily_clusters";')
results = cur.fetchall()
df = pd.DataFrame(results, columns=[desc[0] for desc in cur.description])
print(df.head())

   30pid_superfamily_id  centroid_orf_id date_clustered
0                     1               17     2026-05-26
1                     2               35     2026-05-26
2                     3               37     2026-05-26
3                     4               40     2026-05-26
4                     5               65     2026-05-26


In [13]:
def fetch_sequences(fasta_path, target_ids):
    target_ids = set(str(x) for x in target_ids)
    remaining = set(target_ids)
    sequences = {}
    with open(fasta_path) as f:
        orf_id = None
        seq_parts = []
        for line in f:
            if not remaining:
                break
            line = line.strip()
            if line.startswith('>'):
                if orf_id and orf_id in target_ids:
                    sequences[orf_id] = "".join(seq_parts)
                    remaining.discard(orf_id)
                orf_id = line[1:].split('|')[0]
                seq_parts = []
            elif orf_id in target_ids:
                seq_parts.append(line)
        if orf_id and orf_id in target_ids:
            sequences[orf_id] = "".join(seq_parts)
    return sequences

sequences = fetch_sequences("../data/petadex.catalytic_orfs.v1.1.fa", df["centroid_orf_id"].tolist())
df["sequence"] = df["centroid_orf_id"].map(lambda x: sequences.get(str(x)))
print(df[["centroid_orf_id", "sequence"]].head())

   centroid_orf_id                                           sequence
0               17  MELNSMQFLLGLIGLLLLIVTSLRRWLLRRESPQKQAVDFHGELYQ...
1               35  MPTEVVMPKWGLSMQEGKINLWLKREGEAVQKGEPIAEVETEKITN...
2               37  MIRPVTFRNMNQQIIGILHTPDNIRLNEKVPGILMFHGFTGNKTEA...
3               40  MSVSLKGALRVLPLAASVALVGCFGGGDPEPGPDGQALTNPGEYEI...
4               65  MDENYPVLPGADSFFIKGNEIGILISHGFNGTPQSVRFLGRAMASD...


In [18]:
df.to_csv("../data/superfamily_clusters_with_sequences.csv", index=False)